# NLLB native-language finetuning — reproducible experiment

**Edit only the next cell** for each experiment, then run cells from top to bottom. Default: real aligned text files → seeded 80/10/10 split → native NLLB LoRA finetuning. **Development loss and best-checkpoint selection remain ON.** Costly pretraining baselines, test inference and publication reports are OFF. The original input files are never modified.

For a quick example test, change only `data_mode` to `"example"` (Bhojpuri–Hindi) and `max_steps` to `1`. For a single two-column text file, choose `"tsv"`. For a previous JSONL corpus, choose `"jsonl"` or `"pre_split"`. Use a **new run_label** for each new model/training configuration. See `docs/experiment-guide.md` and `docs/notebook-platforms.md`.


## 1. Experiment settings — edit here only

All flags and paths are in this one cell. Restart and reimport the notebook for every experiment as planned.


In [ ]:
# EDIT ONLY THIS CELL FOR A NEW NLLB EXPERIMENT.
# A fresh Kaggle/Colab session and a freshly imported notebook are assumed.
EXPERIMENT = {
    # DATA: two aligned text files (default); or "tsv", "jsonl", "pre_split", "example".
    "data_mode": "aligned_text",
    "source_lang": "bho_Deva",        # Native NLLB language token.
    "target_lang": "hin_Deva",
    "source_text": "/kaggle/input/YOUR_DATASET/bho.txt",  # One sentence per line.
    "target_text": "/kaggle/input/YOUR_DATASET/hin.txt",  # Same number/order of lines.
    "tsv_file": "/kaggle/input/YOUR_DATASET/parallel.tsv", # For data_mode="tsv" (source<TAB>target).
    "tsv_has_header": False,
    "parallel_jsonl": "data/local/bho-hin/parallel.jsonl", # For data_mode="jsonl".
    "split_directory": "data/local/bho-hin/splits",       # For data_mode="pre_split".
    "dataset_id": "corpus-v1",       # Change when replacing the input files.
    "split_seed": 42,              # Same seed/data split for fair model comparisons.
    "split_ratios": (0.80, 0.10, 0.10),
    "group_by_document": True,     # Keep repeated sentences and documents in one split.
    "validate_data": False,       # False: fast start; True: full exact leakage audit.
    "data_report": False,         # Optional publication corpus plots (not needed for training).

    # MODEL / TRAINING. Dev loss is ALWAYS computed during training, each epoch.
    "model_id": "facebook/nllb-200-distilled-600M",
    "model_revision": None,        # Optional fixed HF commit hash.
    "conditioning": "native",     # Default; "text_prefix"/"dedicated_token" are research modes.
    "extend_language": None,      # E.g. "bra_Deva" (requires dedicated_token and full training).
    "language_donor": "hin_Deva",
    "proxy_source_lang": None,     # Only for explicit text_prefix experiments.
    "proxy_target_lang": None,
    "finetune_mode": "lora",       # "full" uses much more GPU memory.
    "train_seed": 42,
    "epochs": 3,
    "max_steps": -1,               # -1 = full epochs; set 1 for a software-only smoke test.
    "batch_size": 1,
    "gradient_accumulation_steps": 16,
    "learning_rate": 2e-4,        # Use ~2e-5 when changing to full finetuning.
    "max_length": 128,
    "max_new_tokens": 128,
    "beams": 4,
    "precision": "fp16",          # T4: fp16; bf16 requires supported GPU; fp32 slower.
    "gradient_checkpointing": True,
    "lora_rank": 8,
    "run_label": "pilot-1",        # Change for every independent training experiment.
    "allow_training": True,       # The requested real finetuning run.

    # OPTIONAL COMPARISONS / PUBLICATION REPORTS. All are OFF for the pilot.
    "pretrain_baseline": False,   # Extra full-model inference on held-out DEV sample.
    "baseline_dev_size": 100,
    "posttrain_test": False,      # Test predictions only AFTER model selection/training.
    "publication_report": False,  # BLEU/chrF++, plots, error-review sheets; requires posttrain_test.
    "test_baseline": False,       # Additional expensive untouched-model test inference.
    "save_archive": True,        # Package exported model/config into a Kaggle output ZIP.
    "include_checkpoints": False,# True only to resume optimizer state (large).
    "setup_mode": "train",        # GPU training: "train"; CPU inspection: "base".
    "refresh_repo": False,       # True runs git pull --ff-only on cloned repo.
}


## 2. Prepare environment

Select GPU in the notebook UI first. This cell clones or optionally updates the repo and installs model dependencies; it has no experimental flags.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json

# Install/clone AFTER the settings cell. No separate terminal is needed.
ROOT = (Path("/kaggle/working/BhaashikMTFinetuning")
        if Path("/kaggle/working").is_dir()
        else Path("/content/BhaashikMTFinetuning")
        if Path("/content").is_dir() else Path.cwd().resolve())
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
if not (ROOT / "pyproject.toml").is_file():
    if ROOT.exists():
        raise ValueError(f"Checkout exists but is incomplete: {ROOT}")
    subprocess.run(["git", "clone",
                    "https://github.com/bhaashik/BhaashikMTFinetuning.git",
                    str(ROOT)], check=True)
elif EXPERIMENT["refresh_repo"]:
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
os.chdir(ROOT)
loaded = [x for x in ("torch", "transformers", "sentencepiece", "datasets", "peft")
          if x in sys.modules]
if loaded:
    raise RuntimeError("Restart session before dependency installation: "+", ".join(loaded))
setup_mode = EXPERIMENT["setup_mode"]
command = [sys.executable, str(ROOT / "scripts" / "setup_notebook.py"),
           "--extras", setup_mode]
if setup_mode.startswith("train"):
    command += ["--require-cuda"]
subprocess.run(command, check=True)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
from bhaashik.experiment_workflow import check_settings
check_settings(EXPERIMENT, model_family="nllb")
print("Checkout:", ROOT, "| mode:", setup_mode)


## 3. Convert and split parallel data

Reads the configured files (including `/kaggle/input` or Google Drive), creates tagged JSONL and three reproducible splits. Full validation and plots are optional. Displays paths and split counts.


In [ ]:
from bhaashik.experiment_workflow import (
    prepare_parallel_data, experiment_paths, report_data_if_enabled,
    training_parameters, write_experiment_config,
)

PREPARED = prepare_parallel_data(ROOT, EXPERIMENT, model_family="nllb")
PATHS = experiment_paths(ROOT, EXPERIMENT, family="nllb", prepared=PREPARED)
DATA = PREPARED["data_dir"]     # Read train.jsonl, dev.jsonl, test.jsonl here.
RUN = PATHS["run"]             # Exported model is written here.
REPORT_DIR = PATHS["report"]
report_data_if_enabled(EXPERIMENT, PREPARED, REPORT_DIR)
MODEL_ID = EXPERIMENT["model_id"]
if EXPERIMENT["extend_language"]:
    from bhaashik.modeling import extend
    assert EXPERIMENT["conditioning"] == "dedicated_token"
    assert EXPERIMENT["finetune_mode"] == "full"
    extended = ROOT / "runs" / f"language-extension-{EXPERIMENT['extend_language']}"
    if not (extended / "bhaashik_extension.json").is_file():
        extend(MODEL_ID, extended, EXPERIMENT["extend_language"],
               donor=EXPERIMENT["language_donor"],
               revision=EXPERIMENT["model_revision"])
    MODEL_ID = str(extended)
TRAIN_CONFIG = training_parameters(EXPERIMENT, PREPARED, PATHS,
                                   model_family="nllb")
TRAIN_CONFIG["model"] = MODEL_ID
if EXPERIMENT["extend_language"]:
    TRAIN_CONFIG["revision"] = None
TRAIN_CONFIG_PATH = write_experiment_config(
    EXPERIMENT, PREPARED, PATHS, TRAIN_CONFIG)
print("Run:", RUN, "\nConfig:", TRAIN_CONFIG_PATH)


## 4. Optional untouched-model development baseline

**Skipped by default.** Turn on `pretrain_baseline` in cell 1 only when needed for a paper. Development-set loss during training does **not** require this baseline.


In [ ]:
from bhaashik.modeling import translate
from bhaashik.model_comparisons import select_dev_sample
from bhaashik.pretrain_review import compare_dev, plot_review
from bhaashik.data import file_hash

if EXPERIMENT["pretrain_baseline"]:
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    sample = REPORT_DIR / "dev_sample.jsonl"
    if not sample.exists():
        select_dev_sample(DATA / "dev.jsonl", sample,
                          sample_size=EXPERIMENT["baseline_dev_size"],
                          seed=EXPERIMENT["split_seed"])
    else:
        manifest = json.loads(Path(str(sample)+".manifest.json").read_text())
        if manifest["source_sha256"] != file_hash(DATA / "dev.jsonl"):
            raise ValueError("Dev sample is from different data. Change run_label.")
    predictions = REPORT_DIR / "nllb-original-dev.jsonl"
    if not predictions.exists():
        translate(MODEL_ID, EXPERIMENT["source_lang"], EXPERIMENT["target_lang"],
                  sample, predictions, revision=TRAIN_CONFIG["revision"],
                  conditioning=EXPERIMENT["conditioning"],
                  proxy_source_lang=EXPERIMENT["proxy_source_lang"],
                  proxy_target_lang=EXPERIMENT["proxy_target_lang"],
                  batch_size=EXPERIMENT["batch_size"],
                  max_length=EXPERIMENT["max_length"],
                  max_new_tokens=EXPERIMENT["max_new_tokens"],
                  beams=EXPERIMENT["beams"])
    print(compare_dev(sample, {"NLLB original": predictions}, REPORT_DIR))
    if EXPERIMENT["publication_report"]:
        print("Figures:", plot_review(REPORT_DIR))
else:
    print("Costly pre-training baseline skipped (default).")


## 5. Finetune NLLB

Normal training computes **validation/development loss each epoch**, retains the best checkpoint and exports the merged LoRA model. Use `max_steps=1` only to test the software.


In [ ]:
if EXPERIMENT["allow_training"]:
    assert EXPERIMENT["setup_mode"].startswith("train"), (
        "Set setup_mode='train' and select a GPU in Kaggle/Colab.")
    from bhaashik.modeling import train
    # The trainer checks train/dev leakage and computes evaluation LOSS each epoch.
    # It restores the checkpoint with the lowest development loss.
    DEV_LOSS = train(TRAIN_CONFIG_PATH)
    print("Training complete. Development loss:", DEV_LOSS)
    print("Exported model:", RUN)
else:
    print("Training disabled. Set allow_training=True in the FIRST cell.")


## 6. Optional final test inference and publication reports

**Off by default.** After model selection, enable `posttrain_test` and optionally `publication_report`. The test set is never used to tune training settings.


In [ ]:
if EXPERIMENT["posttrain_test"]:
    assert (RUN / "run_manifest.json").is_file(), "Train first."
    from bhaashik.modeling import translate
    prediction_path = RUN / "predictions.jsonl"
    if not prediction_path.exists():
        translate(RUN, EXPERIMENT["source_lang"], EXPERIMENT["target_lang"],
                  DATA / "test.jsonl", prediction_path,
                  conditioning=EXPERIMENT["conditioning"],
                  proxy_source_lang=EXPERIMENT["proxy_source_lang"],
                  proxy_target_lang=EXPERIMENT["proxy_target_lang"],
                  batch_size=EXPERIMENT["batch_size"],
                  max_length=EXPERIMENT["max_length"],
                  max_new_tokens=EXPERIMENT["max_new_tokens"],
                  beams=EXPERIMENT["beams"])
    print("Held-out test predictions:", prediction_path)
    baseline_prediction = None
    if EXPERIMENT["test_baseline"]:
        baseline_prediction = REPORT_DIR / "nllb-original-test.jsonl"
        REPORT_DIR.mkdir(parents=True, exist_ok=True)
        if not baseline_prediction.exists():
            translate(MODEL_ID, EXPERIMENT["source_lang"], EXPERIMENT["target_lang"],
                      DATA / "test.jsonl", baseline_prediction,
                      revision=TRAIN_CONFIG["revision"],
                      conditioning=EXPERIMENT["conditioning"],
                      proxy_source_lang=EXPERIMENT["proxy_source_lang"],
                      proxy_target_lang=EXPERIMENT["proxy_target_lang"],
                      batch_size=EXPERIMENT["batch_size"],
                      max_length=EXPERIMENT["max_length"],
                      max_new_tokens=EXPERIMENT["max_new_tokens"],
                      beams=EXPERIMENT["beams"])
    if EXPERIMENT["publication_report"]:
        from bhaashik.evaluation import evaluate, export_errors
        result = evaluate(prediction_path, RUN / "evaluation.json",
                          baseline=baseline_prediction, samples=1000,
                          seed=EXPERIMENT["train_seed"])
        print("Final test metrics:", result["scores"])
        export_errors(prediction_path, RUN / "annotations.csv")
        from bhaashik.pretrain_review import plot_review
        print("Additional figures:", plot_review(REPORT_DIR))
else:
    print("Test inference/reporting OFF. Enable posttrain_test after model selection.")


## 7. Preserve outputs

Export current model, config and optional reports in a ZIP. Large optimizer/checkpoint snapshots are omitted unless `include_checkpoints=True`. Save outputs to Kaggle version or Google Drive before the runtime ends.


In [ ]:
from bhaashik.experiment_workflow import save_optional_archive
if EXPERIMENT["save_archive"]:
    assert (RUN / "run_manifest.json").is_file(), "Finish training before archiving."
archive = save_optional_archive(
    ROOT, EXPERIMENT, PATHS,
    report=REPORT_DIR if REPORT_DIR.is_dir() else None)
if archive:
    print("Save/download this output ZIP:", archive)
    print("MiB:", round(archive.stat().st_size / 2**20, 1))
# Kaggle: right-side Output/Files panel or Save Version with outputs.
# Colab: copy the ZIP from /content/ to mounted Google Drive.
